# Symbol Analysis

Parametrized notebook for per-symbol analysis of Italian market stocks.
Uses pre-computed `analysis_results.parquet` (relative signals vs FTSEMIB.MI).

**Sections:**
1. Data Loading
2. Price Overview (absolute vs relative)
3. Regime Analysis
4. Signal Visualization (Breakout, Turtle, MA)
5. Performance per Signal (P&L, cumulative returns)

In [ ]:
# papermill parameter cell — override SYMBOL when generating per-symbol notebooks
SYMBOL = "ENI.MI"

In [ ]:
import logging
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

from algoshort.plots import (
    plot_abs_rel,
    plot_price_signal_cumreturns,
    plot_profit_loss,
    plot_regime_rel,
    plot_signal_bo,
    plot_signal_ma,
    plot_signal_rel,
    plot_signal_tt,
)

logging.basicConfig(
    level=logging.WARNING,
    format="%(asctime)s - %(name)s - %(levelname)s - %(message)s",
)

%matplotlib inline
plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams["figure.figsize"] = (14, 6)
plt.rcParams["font.size"] = 10

In [ ]:
# --- Paths (relative to project root) ---
ANALYSIS_PATH = Path("data/results/it/analysis_results.parquet")
BENCHMARK = "FTSEMIB.MI"

# Signals available in analysis_results and their display labels.
# Each entry: (column_name, display_label)
# Note: all signals are relative (computed vs FTSEMIB.MI benchmark).
SIGNALS = [
    ("rbo_20",       "BO-20"),
    ("rbo_50",       "BO-50"),
    ("rbo_150",      "BO-150"),
    ("rtt_5020",     "TT-50/20"),
    ("rsma_50100",   "SMA-50/100"),
    ("rsma_100150",  "SMA-100/150"),
    ("rsma_50100150","SMA-50/100/150"),
    ("rema_50100",   "EMA-50/100"),
    ("rema_100150",  "EMA-100/150"),
    ("rema_50100150","EMA-50/100/150"),
]

In [ ]:
# --- Data loading ---
_raw = pd.read_parquet(ANALYSIS_PATH)
df = _raw[_raw["symbol"] == SYMBOL].copy().reset_index(drop=True)

if df.empty:
    raise ValueError(
        f"No data found for symbol '{SYMBOL}' in {ANALYSIS_PATH}. "
        f"Available symbols: {sorted(_raw['symbol'].unique().tolist())}"
    )

# Set date as index — required for correct x-axis in all plot functions.
df = df.set_index("date")

print(f"Symbol : {SYMBOL}")
print(f"Rows   : {len(df)}")
print(f"Range  : {df.index.min().date()} → {df.index.max().date()}")
print(f"Columns: {len(df.columns)}")

## 1. Price Overview

Absolute close price vs relative close (rebased to FTSEMIB.MI benchmark).

In [ ]:
# Columns: close (absolute), rclose (relative to benchmark)
plot_abs_rel(df[["close", "rclose"]], ticker=SYMBOL, bm_name=BENCHMARK);

## 2. Regime Analysis

Floor/ceiling signal detection and regime state (relative vs FTSEMIB.MI).

In [ ]:
# Relative floor/ceiling signal
# Columns: rclose, rh3, rl3, rclg, rflr, rrg_ch, rrg
_regime_cols = ["rclose", "rh3", "rl3", "rclg", "rflr", "rrg_ch", "rrg"]
plot_signal_rel(df[_regime_cols], ticker=SYMBOL);

In [ ]:
# Relative regime detection
# Columns: rclose, rrg, rl3, rh3, rclg, rflr, rrg_ch
_regime_cols2 = ["rclose", "rrg", "rl3", "rh3", "rclg", "rflr", "rrg_ch"]
plot_regime_rel(df[_regime_cols2], ticker=SYMBOL);

## 3. Breakout Signals

Relative breakout signals for windows 20, 50, and 150 bars.
Columns: `rclose`, `rhi_{window}`, `rlo_{window}`, `rbo_{window}`.

In [ ]:
for _window in [20, 50, 150]:
    _bo_cols = ["rclose", f"rhi_{_window}", f"rlo_{_window}", f"rbo_{_window}"]
    _missing = [c for c in _bo_cols if c not in df.columns]
    if _missing:
        print(f"Skipping BO-{_window}: missing columns {_missing}")
        continue
    plot_signal_bo(df[_bo_cols], window=_window, ticker=SYMBOL, relative=True);

## 4. Turtle Trading Signal (slow=50, fast=20)

Relative turtle signal. `rclose` is used as price axis; `rtt_5020` is renamed
to `turtle_5020` to match the naming convention expected by `plot_signal_tt`.

In [ ]:
# plot_signal_tt expects: close, turtle_{slow}{fast}
# We use rclose as 'close' and rename the relative signal column.
_tt_df = df[["rclose", "rtt_5020"]].rename(
    columns={"rclose": "close", "rtt_5020": "turtle_5020"}
)
plot_signal_tt(_tt_df, fast=20, slow=50, ticker=SYMBOL);

## 5. Moving Average Signals (50 / 100 / 150)

Relative SMA and EMA triple-crossover signals. `rclose` is used as price axis;
`rsma_50100150` / `rema_50100150` are renamed to match `plot_signal_ma` expectations.

In [ ]:
# plot_signal_ma expects: close, sma_{st}{mt}{lt}, ema_{st}{mt}{lt}
_ma_df = df[["rclose", "rsma_50100150", "rema_50100150"]].rename(
    columns={
        "rclose": "close",
        "rsma_50100150": "sma_50100150",
        "rema_50100150": "ema_50100150",
    }
)
plot_signal_ma(_ma_df, st=50, mt=100, lt=150, ticker=SYMBOL);

## 6. Performance per Signal

For each signal: cumulative P&L / daily changes and price + signal + cumulative returns.

**Column mapping** (per signal `s`):
| Source column | Renamed to | Plot function |
|---|---|---|
| `{s}_PL_cum` | `tt_PL_cum` | `plot_profit_loss` |
| `{s}_chg1D` | `tt_chg1D` | `plot_profit_loss` |
| `rclose` | `close` | `plot_price_signal_cumreturns` |
| `{s}_stop_loss` | `stop_loss` | `plot_price_signal_cumreturns` |
| `{s}_cumul` | `tt_cumul` | `plot_price_signal_cumreturns` |

In [ ]:
def _prep_profit_loss(df: pd.DataFrame, signal: str) -> pd.DataFrame:
    """Select and rename P&L columns for plot_profit_loss."""
    return df[[f"{signal}_PL_cum", f"{signal}_chg1D"]].rename(
        columns={
            f"{signal}_PL_cum": "tt_PL_cum",
            f"{signal}_chg1D": "tt_chg1D",
        }
    )


def _prep_price_signal_cumreturns(df: pd.DataFrame, signal: str) -> pd.DataFrame:
    """Select and rename columns for plot_price_signal_cumreturns.

    Uses rclose as 'close' so price and stop-loss are on the same relative scale.
    """
    return df[["rclose", f"{signal}_stop_loss", signal, f"{signal}_cumul"]].rename(
        columns={
            "rclose": "close",
            f"{signal}_stop_loss": "stop_loss",
            f"{signal}_cumul": "tt_cumul",
        }
    )

In [ ]:
for _sig, _label in SIGNALS:
    print(f"\n{'='*60}")
    print(f"  Signal: {_label} ({_sig})")
    print(f"{'='*60}")

    # --- P&L plot ---
    _pl_cols = [f"{_sig}_PL_cum", f"{_sig}_chg1D"]
    if all(c in df.columns for c in _pl_cols):
        plot_profit_loss(_prep_profit_loss(df, _sig), ticker=SYMBOL, method=_label)
    else:
        print(f"  [skip] plot_profit_loss: missing {[c for c in _pl_cols if c not in df.columns]}")

    # --- Price + signal + cumulative returns plot ---
    _psc_cols = ["rclose", f"{_sig}_stop_loss", _sig, f"{_sig}_cumul"]
    if all(c in df.columns for c in _psc_cols):
        plot_price_signal_cumreturns(
            _prep_price_signal_cumreturns(df, _sig),
            ticker=SYMBOL,
            signal=_sig,
            method=_label,
        )
    else:
        print(f"  [skip] plot_price_signal_cumreturns: missing {[c for c in _psc_cols if c not in df.columns]}")